<a href="https://colab.research.google.com/github/devarsh-mavani-19/facenet-notebook/blob/main/facenet.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Face Recognition Similarity Search

This notebook demonstrates two approaches for face similarity search:

1. **In‑Memory Search using FAISS**
2. **Vector Database Search using Qdrant**

Both use FaceNet embeddings.

In [ ]:
# Install dependencies
!pip install facenet-pytorch faiss-cpu qdrant-client tqdm matplotlib

In [ ]:
# Load libraries and FaceNet model
import torch
import numpy as np
import os
from PIL import Image
from tqdm import tqdm
from facenet_pytorch import MTCNN, InceptionResnetV1

device = 'cuda' if torch.cuda.is_available() else 'cpu'

mtcnn = MTCNN(image_size=160, margin=20, device=device)

model = InceptionResnetV1(pretrained='vggface2').eval().to(device)

print('FaceNet model loaded')

## Download dataset

In [ ]:
!pip install kaggle

In [ ]:
from google.colab import files
files.upload()

In [ ]:
!mkdir -p ~/.kaggle
!mv kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

In [ ]:
!kaggle datasets download -d hereisburak/pins-face-recognition

In [ ]:
!unzip pins-face-recognition.zip

## Generate embeddings from dataset

In [ ]:
dataset_path = '/content/105_classes_pins_dataset'

embeddings = []
labels = []
image_paths = []

people = sorted(os.listdir(dataset_path))[:5]

for person in tqdm(people):
    person_dir = os.path.join(dataset_path, person)
    if not os.path.isdir(person_dir):
        continue

    for img_name in os.listdir(person_dir):
        img_path = os.path.join(person_dir, img_name)

        try:
            img = Image.open(img_path).convert('RGB')
            face = mtcnn(img)
            if face is None:
                continue

            face = face.unsqueeze(0).to(device)

            with torch.no_grad():
                emb = model(face)

            embeddings.append(emb.cpu().numpy()[0])
            labels.append(person)
            image_paths.append(img_path)

        except:
            pass

embeddings = np.array(embeddings).astype('float32')

print('Total embeddings:', embeddings.shape)

# SECTION A: In‑Memory Vector Search (FAISS)

In [ ]:
import faiss

dimension = 512

faiss.normalize_L2(embeddings)

index = faiss.IndexFlatIP(dimension)
index.add(embeddings)

print('Indexed vectors:', index.ntotal)

In [ ]:
def search_face_faiss(image_path, k=5):

    img = Image.open(image_path).convert('RGB')
    face = mtcnn(img)
    face = face.unsqueeze(0).to(device)

    with torch.no_grad():
        emb = model(face)

    query = emb.cpu().numpy().astype('float32')
    faiss.normalize_L2(query)

    distances, indices = index.search(query, k)

    for i in range(k):
        print(distances[0][i], image_paths[indices[0][i]])

# SECTION B: Vector Database Search (Qdrant)

In [ ]:
from qdrant_client import QdrantClient
from qdrant_client.models import VectorParams, Distance, PointStruct

client = QdrantClient(':memory:')

client.create_collection(
    collection_name='faces',
    vectors_config=VectorParams(size=512, distance=Distance.COSINE)
)

In [ ]:
points = []

for i, emb in enumerate(embeddings):
    points.append(
        PointStruct(
            id=i,
            vector=emb.tolist(),
            payload={
                'person': labels[i],
                'image_path': image_paths[i]
            }
        )
    )

client.upsert(collection_name='faces', points=points)

print('Vectors inserted into Qdrant')

In [ ]:
def search_face_qdrant(image_path, k=5):

    img = Image.open(image_path).convert('RGB')
    face = mtcnn(img)
    face = face.unsqueeze(0).to(device)

    with torch.no_grad():
        emb = model(face)

    query = emb.cpu().numpy()[0]

    results = client.query_points(
        collection_name='faces',
        query=query.tolist(),
        limit=k
    )

    for r in results.points:
        print(r.score, r.payload['person'], r.payload['image_path'])

In [ ]:
search_face_faiss("/content/adriana.jpeg")
search_face_qdrant("/content/adriana.jpeg")